## 3 Omnilingual Evaluation

This notebook evaluates the finetuned Omnilingual model.

In [3]:
# Cell 1: Imports
import torch
from tqdm import tqdm
from omnilingual_asr.data import load_all_data
from omnilingual_asr.evaluate import add_metrics_columns, idiom_summary, print_evaluation_summary, show_examples
from omnilingual_asr.models.inference.pipeline import ASRInferencePipeline
from omnilingual_asr.models.wav2vec2_llama.lang_ids import supported_langs

/local/scratch/matuor/omnilingual/.venv/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [16]:
# Cell 2: Configuration
MODEL_CARD = "../../../../../models/omnilingual-ctc-rm-1b"
LANGUAGE_CODE = "roh_Latn_surs1244"
BATCH_SIZE = 8
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {DEVICE}, Lang supported: {LANGUAGE_CODE in supported_langs}")

Device: cpu, Lang supported: True


In [5]:
# Cell 3: Load test data
df_test = load_all_data("test")
print(f"Loaded {len(df_test)} samples")

Loaded 631 samples


In [17]:
# Cell 4: Transcribe
pipeline = ASRInferencePipeline(model_card=MODEL_CARD, device=DEVICE)
audio_paths = df_test["audio_path"].tolist()
transcriptions = []

for i in tqdm(range(0, len(audio_paths), BATCH_SIZE)):
    batch = audio_paths[i:i+BATCH_SIZE]
    try:
        results = pipeline.transcribe(batch, lang=[LANGUAGE_CODE]*len(batch), batch_size=len(batch))
        transcriptions.extend(results)
    except Exception as e:
        print(f"Batch error at {i}: {e}")
        transcriptions.extend([""] * len(batch))

df_test["omnilingual_transcription"] = transcriptions

ModelNotKnownError: ../../../../../models/omnilingual-ctc-rm-1b is not a known model.

In [ ]:
# Cell 5: Compute metrics
df_test = add_metrics_columns(df_test, "sentence", "omnilingual_transcription")
summary = idiom_summary(df_test)
print_evaluation_summary(summary)

In [ ]:
# Cell 6: Show examples
show_examples(df_test, "omnilingual_transcription")

In [10]:
import pyarrow.dataset as pa_ds
import polars as pl

ds = pa_ds.dataset(
    "../../parquet-dataset/rm-dataset/version=0",
    partitioning="hive",
)
table = ds.to_table(columns=["audio_size", "split"])
df = pl.from_arrow(table).filter(pl.col("split") == "train")

# Count how many samples are longer than 80_000 samples (5 seconds)
long_samples = df.filter(pl.col("audio_size") > 480_000)
print(f"Total train samples: {len(df)}")
print(f"Samples > 5s (80k): {len(long_samples)} ({len(long_samples)/len(df)*100:.1f}%)")

Total train samples: 29565
Samples > 5s (80k): 2145 (7.3%)
